In [ ]:
# Cell 1: Setup
!pip install -q sentence-transformers vstash
!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull origin develop)
%cd /content/vstash

In [ ]:
# Cell 2: HF Login (needed for upload later)
!hf auth login

In [ ]:
# Cell 3: Generate adaptive triples + Train + Evaluate (all in one)
import math
import random
import statistics
import tempfile
from pathlib import Path

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

import sys

sys.path.insert(0, "/content/vstash")
from experiments.beir_benchmark import download_beir, load_beir
from vstash.embed import embed_query, get_embedding_dim
from vstash.store import VstashStore

MODEL = "BAAI/bge-small-en-v1.5"
DATASETS = ["scifact", "nfcorpus", "fiqa", "scidocs", "arguana"]
TOP_K = 10
dim = get_embedding_dim(MODEL)

# ========================================
# STEP 1: Generate triples with adaptive weights
# ========================================
print("=" * 60)
print("  STEP 1: Generating adaptive triples")
print("=" * 60)

all_pairs = []
for ds_name in DATASETS:
    cache = download_beir(ds_name)
    corpus, queries, qrels = load_beir(cache)
    doc_items = list(corpus.items())[:5000]

    db_path = tempfile.mktemp(suffix=".db")
    store = VstashStore(db_path, embedding_dim=dim)

    doc_id_to_path = {}
    for doc_id, doc in doc_items:
        text = doc.get("title", "") + "\n" + doc.get("text", "")
        emb = embed_query(text[:512], MODEL)
        path = f"/{ds_name}/{doc_id}"
        doc_id_to_path[doc_id] = path
        store.add_document(
            path=path,
            title=doc.get("title", doc_id),
            chunks=[text],
            embeddings=[emb],
            source_type="text",
        )

    query_items = [(qid, qt) for qid, qt in queries.items() if qid in qrels]
    pairs = []
    disagree = 0

    for qid, qtext in query_items:
        gold_paths = {
            doc_id_to_path[d] for d in qrels[qid] if qrels[qid][d] > 0 and d in doc_id_to_path
        }
        if not gold_paths:
            continue

        emb = embed_query(qtext, MODEL)
        wc = len(qtext.split())

        # Adaptive weights by query length
        if wc <= 10:
            vh, fh, vl, fl = 0.70, 0.30, 0.30, 0.70
        elif wc <= 50:
            vh, fh, vl, fl = 0.85, 0.15, 0.15, 0.85
        else:
            vh, fh, vl, fl = 0.95, 0.05, 0.50, 0.50

        try:
            vec_r = store.search(
                query_embedding=emb,
                query_text=qtext,
                top_k=TOP_K,
                vec_weight=vh,
                fts_weight=fh,
                adaptive_rrf=False,
            )
            fts_r = store.search(
                query_embedding=emb,
                query_text=qtext,
                top_k=TOP_K,
                vec_weight=vl,
                fts_weight=fl,
                adaptive_rrf=False,
            )
        except Exception:
            continue

        vec_paths = {r.path for r in vec_r[:5]}
        fts_paths = {r.path for r in fts_r[:5]}
        if vec_paths != fts_paths:
            disagree += 1

        doc_texts = {}
        for r in vec_r + fts_r:
            doc_texts[r.path] = r.text

        for gp in gold_paths:
            if gp not in doc_texts:
                continue
            pos = doc_texts[gp]
            pairs.append({"query": qtext, "positive": pos})

    n_q = len(
        [
            q
            for q, _ in query_items
            if any(qrels[q].get(d, 0) > 0 and d in doc_id_to_path for d in qrels[q])
        ]
    )
    print(
        f"  {ds_name}: {n_q} queries with gold, "
        f"{disagree} disagree ({disagree / max(n_q, 1) * 100:.1f}%), "
        f"{len(pairs)} pairs"
    )
    all_pairs.extend(pairs)
    store.close()
    Path(db_path).unlink(missing_ok=True)

random.Random(42).shuffle(all_pairs)
print(f"\nTotal: {len(all_pairs)} pairs")

In [ ]:
# Cell 4: Train MNRL v4 (adaptive weights)
print("\n" + "=" * 60)
print("  STEP 2: Training MNRL v4")
print("=" * 60)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
examples = [InputExample(texts=[p["query"], p["positive"]]) for p in all_pairs]
loader = DataLoader(examples, shuffle=True, batch_size=64)
loss = losses.MultipleNegativesRankingLoss(model)

print(f"  Pairs: {len(all_pairs)}, Epochs: 2, LR: 3e-6, Batch: 64")

model.fit(
    train_objectives=[(loader, loss)],
    epochs=2,
    warmup_steps=50,
    optimizer_params={"lr": 3e-6},
    output_path="experiments/models/vstash-bge-mnrl-v4",
    show_progress_bar=True,
)
model.save("experiments/models/vstash-bge-mnrl-v4")
print("Training done.")

In [ ]:
# Cell 5: Evaluate on all 5 BEIR datasets (full pipeline)
print("\n" + "=" * 60)
print("  STEP 3: Full pipeline evaluation")
print("=" * 60)

BASELINES = {
    "scifact": {"BM25": 0.665, "ColBERTv2": 0.693},
    "nfcorpus": {"BM25": 0.325, "ColBERTv2": 0.344},
    "fiqa": {"BM25": 0.236, "ColBERTv2": 0.356},
    "scidocs": {"BM25": 0.158, "ColBERTv2": 0.154},
    "arguana": {"BM25": 0.315, "ColBERTv2": 0.463},
}

MODELS_EVAL = {
    "BGE-small (base)": "BAAI/bge-small-en-v1.5",
    "vstash-bge-rrf-v4": "experiments/models/vstash-bge-mnrl-v4",
}


def ndcg_at_k(ranked_ids, qr, k=10):
    gains = [qr.get(did, 0) for did in ranked_ids[:k]]
    ideal = sorted(qr.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains[:k]))
    return dcg / idcg if idcg > 0 else 0.0


all_results = {}
for ds_name in DATASETS:
    cache = download_beir(ds_name)
    corpus, queries, qrels = load_beir(cache)
    test_qids = list(qrels.keys())
    print(f"\n  {ds_name}: {len(corpus)} docs, {len(test_qids)} queries")
    all_results[ds_name] = {}

    for label, model_path in MODELS_EVAL.items():
        st_model = SentenceTransformer(model_path)
        d = st_model.get_sentence_embedding_dimension()
        db_path = tempfile.mktemp(suffix=".db")
        store = VstashStore(db_path, embedding_dim=d)

        # Batch embed
        doc_ids = list(corpus.keys())
        doc_texts = [
            (corpus[x].get("title", "") + "\n" + corpus[x].get("text", "")).strip()[:512]
            for x in doc_ids
        ]
        all_embs = st_model.encode(doc_texts, show_progress_bar=True, batch_size=256)

        doc_id_map = {}
        batch = []
        for doc_id, text, emb_arr in zip(doc_ids, doc_texts, all_embs):
            path = f"/beir/{doc_id}"
            doc_id_map[path] = doc_id
            batch.append(
                {
                    "path": path,
                    "title": corpus[doc_id].get("title", doc_id),
                    "chunks": [text],
                    "embeddings": [emb_arr.tolist()],
                    "source_type": "text",
                }
            )
            if len(batch) >= 500:
                store.add_documents_batch(batch)
                batch = []
        if batch:
            store.add_documents_batch(batch)

        ndcgs = []
        for qid in test_qids:
            qemb = st_model.encode(queries[qid]).tolist()
            results = store.search(query_embedding=qemb, query_text=queries[qid], top_k=10)
            ranked = [doc_id_map.get(r.path, "") for r in results]
            ndcgs.append(ndcg_at_k(ranked, qrels[qid], 10))

        mean_ndcg = statistics.mean(ndcgs)
        print(f"    {label:>25}: NDCG@10={mean_ndcg:.4f}")
        all_results[ds_name][label] = mean_ndcg
        store.close()
        Path(db_path).unlink(missing_ok=True)

# Summary
print(f"\n{'=' * 80}")
print("  RESULTS: v4 (adaptive weights) vs base")
print(f"{'=' * 80}")
print(
    f"{'Dataset':>12} {'BM25':>8} {'ColBERTv2':>10} {'Base':>10} {'v4 Tuned':>10} {'Delta':>8} {'vs ColBERT':>10}"
)
print(f"  {'-' * 70}")
for ds in DATASETS:
    bm25 = BASELINES[ds]["BM25"]
    colbert = BASELINES[ds]["ColBERTv2"]
    base = all_results[ds].get("BGE-small (base)", 0)
    tuned = all_results[ds].get("vstash-bge-rrf-v4", 0)
    delta = (tuned - base) / base * 100 if base > 0 else 0
    vs_col = (tuned - colbert) / colbert * 100 if colbert > 0 else 0
    print(
        f"{ds:>12} {bm25:>8.3f} {colbert:>10.3f} {base:>10.4f} {tuned:>10.4f} {delta:>+7.1f}% {vs_col:>+9.1f}%"
    )

In [ ]:
# Cell 6: Upload to HuggingFace (only if results are good)
!hf upload Stffens/bge-small-rrf-v2 experiments/models/vstash-bge-mnrl-v4 --commit-message 'v2: adaptive weights, 5 BEIR datasets'